In [1]:
%%sql

CREATE DATABASE IF NOT EXISTS bronze;

26/06/10 11:22:36 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


++
||
++
++

In [2]:
%%sql 

CREATE DATABASE IF NOT EXISTS silver;

++
||
++
++

In [3]:
%%sql 

CREATE DATABASE IF NOT EXISTS gold;

++
||
++
++

In [4]:
%%sql

CREATE TABLE IF NOT EXISTS bronze.users (
    id BIGINT,
    first_name STRING,
    last_name STRING,
    email STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(created_at))
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Dimension table for user information'
);

++
||
++
++

In [5]:
%%sql

CREATE TABLE IF NOT EXISTS bronze.items (
    id BIGINT,
    name STRING,
    category STRING,
    price DECIMAL(7,2),
    inventory INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (category)
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Dimension table for product items'
);

++
||
++
++

In [6]:
%%sql

CREATE TABLE IF NOT EXISTS bronze.purchases (
    id BIGINT,
    user_id BIGINT,
    item_id BIGINT,
    quantity INT,
    purchase_price DECIMAL(12,2),
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(created_at))
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Fact table for purchase transactions'
);

++
||
++
++

In [7]:
%%sql

CREATE TABLE IF NOT EXISTS bronze.pageviews (
    user_id BIGINT,
    url STRING,
    channel STRING,
    received_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(received_at))
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Fact table for purchase transactions'
);

++
||
++
++

In [8]:
%%sql

CREATE TABLE IF NOT EXISTS silver.users (
    id BIGINT,
    first_name STRING,
    last_name STRING,
    email STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    valid_email BOOLEAN,
    full_name STRING
)
USING iceberg
PARTITIONED BY (days(created_at))
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Validated dimension table for user information'
);

++
||
++
++

In [9]:
%%sql
    
CREATE TABLE IF NOT EXISTS silver.items (
    id BIGINT,
    name STRING,
    category STRING,
    price DECIMAL(7,2),
    inventory INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (category)
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Dimension table for product items'
);

++
||
++
++

In [10]:
%%sql
    
CREATE TABLE IF NOT EXISTS silver.purchases_enriched (
    id BIGINT,
    user_id BIGINT,
    item_id BIGINT,
    quantity INT,
    purchase_price DECIMAL(12,2),
    total_price DECIMAL(14,2),         
    user_email STRING,                
    item_name STRING,
    item_category STRING,
    purchase_date DATE,
    purchase_hour INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP                  
)
USING iceberg
PARTITIONED BY (days(created_at))
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Validated and enriched fact table for purchase transactions'
);

++
||
++
++

In [11]:
%%sql
    
CREATE TABLE IF NOT EXISTS silver.pageviews_by_items (
    user_id BIGINT,
    item_id BIGINT,
    page STRING,
    item_name STRING,
    item_category STRING,
    channel STRING,
    received_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(received_at))
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Fact table for purchase transactions'
);

++
||
++
++

In [12]:
## Validate Table Creation

In [13]:
%%sql
-- List all dbs (namespaces)
SHOW DATABASES;

namespace
bronze
silver
gold


In [14]:
%%sql
-- List all tables in bronze namespace
SHOW TABLES IN bronze;

namespace,tableName,isTemporary
bronze,items,False
bronze,pageviews,False
bronze,purchases,False
bronze,users,False


In [15]:
%%sql
-- List all tables in silver namespace
SHOW TABLES IN silver;

namespace,tableName,isTemporary
silver,items,False
silver,pageviews_by_items,False
silver,purchases_enriched,False
silver,users,False


In [16]:
%%sql
-- Examine the schema of a specific table
DESCRIBE TABLE bronze.users;

col_name,data_type,comment
id,bigint,None
first_name,string,None
last_name,string,None
email,string,None
created_at,timestamp,None
updated_at,timestamp,None
,,
# Partitioning,,
Part 0,days(created_at),


In [17]:
%%sql
-- Examine the schema of a specific table
DESCRIBE TABLE silver.purchases_enriched;

col_name,data_type,comment
id,bigint,None
user_id,bigint,None
item_id,bigint,None
quantity,int,None
purchase_price,"decimal(12,2)",None
total_price,"decimal(14,2)",None
user_email,string,None
item_name,string,None
item_category,string,None
purchase_date,date,None
